# TensorFlow for AI in Fluids  
## From arrays to tensors, gradients, losses, and small neural networks

This notebook is the second programming file for Week 1.  
The previous notebook introduced Python and NumPy for CFD. This notebook introduces **TensorFlow** as the bridge from CFD fields to machine learning.

The goal is not to train a large neural operator yet. The goal is to learn the basic language of machine learning computations:

| TensorFlow idea | Why it matters in AI | Why it matters in fluids |
|---|---|---|
| tensor | stores data and model inputs | stores $u,v,p,\omega,T$ fields |
| shape | tells the structure of data | tells grid size and number of channels |
| dtype | controls numerical precision | affects stability and memory |
| `tf.Variable` | trainable parameter | learned model weights |
| loss | error to minimize | analogous to CFD residual |
| gradient | direction to update parameters | basis of learning and PINNs |
| Keras model | neural network | surrogate, closure, pressure operator |
| batch dimension | multiple samples | multiple flow snapshots |

By the end, students should understand why CFD data naturally become tensors and how TensorFlow computes losses and gradients.

## 0. How to use this notebook

Read the explanation before running each cell.  
For each code cell, ask:

1. What are the inputs?
2. What are the outputs?
3. What is the shape of the tensor?
4. What physical or ML quantity could this represent?
5. What would happen if I changed one parameter?

This habit is important because most bugs in CFD/ML workflows are not syntax errors. They are shape errors, scaling errors, or physical interpretation errors.

In [ ]:
# In Google Colab TensorFlow is usually already installed.
# If you are using a local environment and TensorFlow is missing, install it separately.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Physical devices:", tf.config.list_physical_devices())

## 1. From NumPy arrays to TensorFlow tensors

A TensorFlow tensor is similar to a NumPy array, but TensorFlow also knows how to:
- run tensor operations efficiently,
- use GPU/TPU hardware when available,
- track gradients for learning.

In CFD language, a tensor can store a field such as $u(x,y)$.  
In ML language, a tensor can store inputs, outputs, parameters, or training data.

In [ ]:
# A NumPy array
a_np = np.array([1.0, 2.0, 3.0])

# Convert to TensorFlow tensor
a_tf = tf.constant(a_np)

print("NumPy array:", a_np)
print("TensorFlow tensor:", a_tf)
print("Tensor shape:", a_tf.shape)
print("Tensor dtype:", a_tf.dtype)

### Tensor shape

The **shape** tells how many numbers are stored and how they are arranged.

Examples:
- shape `(N,)`: one-dimensional vector
- shape `(Ny, Nx)`: two-dimensional CFD scalar field
- shape `(Ny, Nx, 2)`: two velocity channels, $u$ and $v$
- shape `(batch, Ny, Nx, channels)`: many CFD snapshots for neural-network training

Shape awareness is one of the most important programming skills for AI in fluids.

In [ ]:
# A 2D CFD-like field
N = 65
x = tf.linspace(0.0, 1.0, N)
y = tf.linspace(0.0, 1.0, N)

X, Y = tf.meshgrid(x, y)
phi = tf.sin(np.pi * X) * tf.sin(np.pi * Y)

print("x shape:", x.shape)
print("X shape:", X.shape)
print("phi shape:", phi.shape)

In [ ]:
plt.figure(figsize=(5, 4))
plt.contourf(X.numpy(), Y.numpy(), phi.numpy(), levels=30)
plt.colorbar(label="phi")
plt.xlabel("x")
plt.ylabel("y")
plt.title("TensorFlow scalar field")
plt.tight_layout()
plt.show()

## 2. Constants and variables

A `tf.constant` is fixed.  
A `tf.Variable` can be changed and can be trained.

In machine learning, weights and biases are variables.  
In a CFD code, a field that is updated during iteration is also conceptually a variable, although in TensorFlow we usually reserve `tf.Variable` for trainable parameters.

In [ ]:
c = tf.constant(3.0)
w = tf.Variable(0.5)

print("constant c:", c)
print("variable w before:", w)

w.assign(2.0)

print("variable w after:", w)

## 3. Basic tensor operations

TensorFlow operations look like mathematical operations.  
This is why TensorFlow is useful for scientific machine learning.

Here we build a small field:

$$
f(x,y) = \sin(\pi x)\sin(\pi y).
$$

This is not yet CFD, but it looks like a smooth mode that could appear in a flow field.

In [ ]:
f = tf.sin(np.pi * X) * tf.sin(np.pi * Y)
g = X**2 + Y**2
h = f + 0.1 * g

print("f min/max:", float(tf.reduce_min(f)), float(tf.reduce_max(f)))
print("h min/max:", float(tf.reduce_min(h)), float(tf.reduce_max(h)))

## 4. Broadcasting

Broadcasting allows TensorFlow to combine tensors with compatible shapes.

For example:
- a scalar can multiply a full field,
- a 1D vector can be added across one direction,
- a channel weight can scale each field channel.

Broadcasting is powerful, but it can also hide mistakes. Always check shapes.

In [ ]:
field = tf.ones((4, 5))
scale = tf.constant(2.0)

print("field shape:", field.shape)
print("scale shape:", scale.shape)
print("result shape:", (scale * field).shape)
print(scale * field)

## 5. Stacking channels: CFD fields as ML inputs

A neural network often expects multiple channels.  
For the cavity problem, a future ML model might use channels such as:

$$
[u(x,y),\; v(x,y),\; \omega(x,y)]
$$

or

$$
[\nabla\cdot u^*,\; p]
$$

For TensorFlow/Keras convolutional networks, the usual shape is:

$$
(\text{batch}, N_y, N_x, \text{channels}).
$$

In [ ]:
u = tf.sin(np.pi * X) * tf.sin(np.pi * Y)
v = -tf.sin(np.pi * X) * tf.sin(np.pi * Y)
omega = tf.cos(np.pi * X) * tf.cos(np.pi * Y)

# Stack as channels: shape (Ny, Nx, 3)
sample = tf.stack([u, v, omega], axis=-1)

# Add batch dimension: shape (1, Ny, Nx, 3)
batch = sample[None, ...]

print("single sample shape:", sample.shape)
print("batch shape:", batch.shape)

## 6. Loss functions

A loss is a number that measures error.  
In supervised learning, a common loss is mean squared error:

$$
\mathcal{L} =
\frac{1}{N}\sum_i (y_i^{pred}-y_i^{true})^2.
$$

This is conceptually similar to a CFD residual.  
A residual measures how badly a field violates the governing equation.  
A loss measures how badly a model matches data or physics.

In [ ]:
y_true = tf.constant([1.0, 2.0, 3.0])
y_pred = tf.constant([0.8, 2.2, 2.9])

mse = tf.reduce_mean((y_pred - y_true)**2)

print("MSE loss:", float(mse))

## 7. Automatic differentiation with GradientTape

TensorFlow can compute derivatives automatically.  
This is essential for training neural networks and physics-informed neural networks.

Example:

$$
y = x^2,\qquad \frac{dy}{dx}=2x.
$$

We ask TensorFlow to compute the derivative.

In [ ]:
x0 = tf.Variable(3.0)

with tf.GradientTape() as tape:
    y0 = x0**2

dy_dx = tape.gradient(y0, x0)

print("x =", float(x0))
print("y = x^2 =", float(y0))
print("dy/dx =", float(dy_dx))

### Gradient of a loss with respect to trainable parameters

In machine learning we usually do not differentiate with respect to $x$.  
We differentiate the loss with respect to model parameters.

Here the model is:

$$
y = ax+b.
$$

TensorFlow will compute $\partial \mathcal{L}/\partial a$ and $\partial \mathcal{L}/\partial b$.

In [ ]:
# Training data
x_data = tf.constant(np.linspace(0, 1, 50), dtype=tf.float32)
y_data = 2.0 * x_data + 0.5

# Trainable parameters
a = tf.Variable(0.0)
b = tf.Variable(0.0)

with tf.GradientTape() as tape:
    y_pred = a * x_data + b
    loss = tf.reduce_mean((y_pred - y_data)**2)

grad_a, grad_b = tape.gradient(loss, [a, b])

print("loss:", float(loss))
print("grad_a:", float(grad_a))
print("grad_b:", float(grad_b))

## 8. Manual training loop

A training loop repeats:
1. predict,
2. compute loss,
3. compute gradients,
4. update parameters.

This is structurally similar to a CFD iteration:
1. compute field update,
2. compute residual,
3. correct the field,
4. repeat.

In [ ]:
a = tf.Variable(0.0)
b = tf.Variable(0.0)
learning_rate = 0.5

loss_history = []

for epoch in range(100):
    with tf.GradientTape() as tape:
        y_pred = a * x_data + b
        loss = tf.reduce_mean((y_pred - y_data)**2)

    grad_a, grad_b = tape.gradient(loss, [a, b])

    a.assign_sub(learning_rate * grad_a)
    b.assign_sub(learning_rate * grad_b)

    loss_history.append(float(loss))

print(f"learned a = {float(a):.4f}")
print(f"learned b = {float(b):.4f}")

plt.figure(figsize=(6, 4))
plt.semilogy(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.grid(alpha=0.3)
plt.title("Manual TensorFlow training loop")
plt.tight_layout()
plt.show()

## 9. Keras: building a small neural network

Keras is the high-level neural-network API inside TensorFlow.

A dense neural network maps inputs to outputs through layers:

$$
x \rightarrow \text{Dense} \rightarrow \text{activation} \rightarrow \text{Dense} \rightarrow y.
$$

Here we train a small network to approximate

$$
f(x)=\sin(2\pi x).
$$

This is a toy example, but it introduces the same workflow used later for flow-field prediction.

In [ ]:
# Make training data
x_train = np.linspace(0, 1, 200).reshape(-1, 1).astype("float32")
y_train = np.sin(2*np.pi*x_train).astype("float32")

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(32, activation="tanh"),
    tf.keras.layers.Dense(32, activation="tanh"),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
              loss="mse")

history = model.fit(x_train, y_train, epochs=300, verbose=0)

print("final loss:", history.history["loss"][-1])

In [ ]:
x_test = np.linspace(0, 1, 300).reshape(-1, 1).astype("float32")
y_test = np.sin(2*np.pi*x_test)
y_nn = model.predict(x_test, verbose=0)

plt.figure(figsize=(6, 4))
plt.plot(x_test, y_test, label="true")
plt.plot(x_test, y_nn, "--", label="neural network")
plt.scatter(x_train[::10], y_train[::10], s=15, label="training points")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.semilogy(history.history["loss"])
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. GradientTape for physics residuals

Physics-informed neural networks use derivatives of the neural-network output.

For example, if

$$
u(x)=\sin(\pi x),
$$

then

$$
\frac{d^2u}{dx^2}+\pi^2\sin(\pi x)=0.
$$

This is a simple differential equation residual.  
We can compute derivatives with TensorFlow.

In [ ]:
x_phys = tf.reshape(tf.linspace(0.0, 1.0, 100), (-1, 1))

with tf.GradientTape(persistent=True) as tape2:
    tape2.watch(x_phys)
    with tf.GradientTape() as tape1:
        tape1.watch(x_phys)
        u_exact = tf.sin(np.pi * x_phys)
    du_dx = tape1.gradient(u_exact, x_phys)
d2u_dx2 = tape2.gradient(du_dx, x_phys)

residual = d2u_dx2 + (np.pi**2) * tf.sin(np.pi * x_phys)

print("maximum physics residual:", float(tf.reduce_max(tf.abs(residual))))

### Why this matters for AI in fluids

Navier--Stokes residuals are more complicated than this 1D residual, but the idea is the same:

1. neural network predicts a field,
2. automatic differentiation computes derivatives,
3. PDE residual is evaluated,
4. residual becomes part of the loss.

This is the foundation of PINNs and many physics-informed models.

## 11. A very small convolutional model for field data

A CFD field is a 2D tensor.  
A convolutional neural network is a natural model for 2D field-to-field maps.

Here we build a tiny CNN that accepts an input with shape:

$$
(N_y,N_x,1)
$$

and outputs another field of the same size.

This is not yet a serious CFD model. It is a shape-and-workflow demonstration.

In [ ]:
N = 32
dummy_input = tf.random.normal((4, N, N, 1))  # batch of 4 scalar fields

cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(N, N, 1)),
    tf.keras.layers.Conv2D(16, kernel_size=3, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(16, kernel_size=3, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(1, kernel_size=3, padding="same")
])

dummy_output = cnn(dummy_input)

print("input shape :", dummy_input.shape)
print("output shape:", dummy_output.shape)

## 12. Required student deliverables

Submit a notebook or PDF export containing:

1. A tensor field plot created with TensorFlow.
2. A short explanation of tensor shape for a CFD field.
3. The manual training loop for $y=ax+b$ and the final values of $a$ and $b$.
4. The Keras sine-function training plot.
5. The automatic-differentiation residual check.
6. A paragraph explaining the similarity between CFD residuals and ML losses.
7. A paragraph explaining why CFD fields can be treated as tensors/images.

This TensorFlow notebook prepares students for the later neural-operator part of the course. It does not replace CFD; it gives the language for learning from CFD data.

## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Foundational or supporting notebook; see ARTICLE_FIGURE_MAP.md for its evidence dependency.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../../ARTICLE_FIGURE_MAP.md).
